In [ ]:
# Load the libraries: yfinance gets stock prices, pandas handles tables, matplotlib draws charts
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

# import and download data for the different canadian stocks at tornto stock exchange.
stocks = ["RY.TO", "TD.TO", "BNS.TO", "SHOP.TO", "ENB.TO", "CNR.TO", "SU.TO", "ATD.TO"]

# Download 2 years of daily closing prices (adjusted for dividends and splits)
data = yf.download(stocks, start="2024-10-01", end="2026-10-01", auto_adjust=True)
prices = data["Close"]

# Save a copy so you don't have to re-download every time
prices.to_csv("prices.csv")

# Show the last 5 rows to check it worked
print(prices.tail())

# How much each price changed from one day to the next, in %
daily_change = prices.pct_change() * 100
daily_change = daily_change.dropna() 

# Total return: how much each stock went up or down over the 2 years
first_price = prices.iloc[0]   # first row = first day
last_price = prices.iloc[-1]   # last row = last day
total_return = (last_price - first_price) / first_price * 100

# Standard deviation
daily_risk = daily_change.std()
yearly_risk = daily_risk * (252 ** 0.5)  # there are about 252 trading days in a year

# Put the results in one table
results = pd.DataFrame()
results["Total return (%)"] = total_return
results["Yearly risk (%)"] = yearly_risk
results = results.sort_values("Total return (%)", ascending=False)
print(results.round(1))

# What $100 put in each stock would be worth over time
value_of_100 = prices / first_price * 100

# Draw one line per stock
value_of_100.plot(figsize=(10, 6))
plt.title("What $100 would be worth")
plt.ylabel("Value ($)")
plt.savefig("growth.png")  # save the chart as a picture
plt.show()


# Correlation: do two stocks go up and down together?
# 1 = always together, 0 = no connection
correlation = daily_change.corr()
print(correlation.round(2))

# Draw it as a colour grid (red = move together, blue = don't)
plt.figure(figsize=(8, 6))
plt.imshow(correlation, cmap="coolwarm")
plt.colorbar()
plt.xticks(range(8), correlation.columns, rotation=45)  # stock names along the bottom
plt.yticks(range(8), correlation.columns)               # stock names down the side
plt.title("How closely the stocks move together")
plt.savefig("correlation.png")
plt.show()

# Portfolio: put the same amount of money in each of the 8 stocks
# Each day, the portfolio's change is the average of the 8 stocks' changes
portfolio_daily_change = daily_change.mean(axis=1)

# Portfolio risk, worked out the same way as before
portfolio_risk = portfolio_daily_change.std() * (252 ** 0.5)

# Average risk of one stock on its own
average_stock_risk = yearly_risk.mean()

print("Portfolio yearly risk:", round(portfolio_risk, 1), "%")
print("Average single-stock risk:", round(average_stock_risk, 1), "%")

